# 01 — Data Preparation and Validation

Extract the local SSA HTML mortality table, perform structural checks, and export the analysis-ready CSV plus tables for the report. Run this notebook first.

## Project setup

Run this notebook from the repository root or from the `notebooks/` directory. All paths are resolved to the repository root. This project uses the `src/` helper modules already present in your repository; the original report did not include their source code.

In [ ]:
from pathlib import Path
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

cwd = Path.cwd().resolve()
project_root = cwd if (cwd / "src").is_dir() else cwd.parent
if not (project_root / "src").is_dir():
    raise FileNotFoundError(
        "Project root not found. Open the repository in VS Code and run the notebook from the repository root or notebooks/."
    )
os.chdir(project_root)
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
report_results_dir = project_root / "results"
report_results_dir.mkdir(parents=True, exist_ok=True)
figures_dir = project_root / "figures"
figures_dir.mkdir(parents=True, exist_ok=True)


## Load and clean the raw mortality table

In [ ]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np

# Quarto will execute from the project root
project_root = Path.cwd()

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.data import load_clean_mortality

raw_path = (
    project_root
    / "data"
    / "raw"
    / "ssa_life_table_2023.html"
)

mortality, rows_removed = load_clean_mortality(raw_path)

print("Cleaned dataset shape:", mortality.shape)
print("Rows removed:", rows_removed)

display(mortality.head())

## Validate the cleaned data

In [ ]:
# Data quality assessment of the cleaned SSA mortality table

from itertools import product

required_columns = ["age", "sex", "qx", "lx", "ex"]

# Expected one record for each age-sex combination
expected_pairs = set(
    product(range(120), ["Male", "Female"])
)

observed_pairs = set(
    mortality[["age", "sex"]].itertuples(
        index=False, name=None
    )
)

missing_pairs = expected_pairs - observed_pairs
unexpected_pairs = observed_pairs - expected_pairs

# Missing and duplicate observations
missing_values = mortality[required_columns].isna().sum()
duplicate_count = mortality.duplicated(
    subset=["age", "sex"]
).sum()

# Numerical and actuarial consistency
valid_qx = mortality["qx"].between(0, 1).all()

valid_life_values = (
    mortality["lx"].ge(0).all()
    and mortality["ex"].ge(0).all()
)

initial_survivors = mortality.loc[
    mortality["age"] == 0, "lx"
]

valid_initial_survivors = (
    len(initial_survivors) == 2
    and initial_survivors.eq(100_000).all()
)

monotone_survivors = all(
    group["lx"].is_monotonic_decreasing
    for _, group in (
        mortality.sort_values("age").groupby("sex")
    )
)

# Generate a summary of the checks
checks = [
    ("Expected number of observations",
     len(mortality) == 240,
     f"{len(mortality)} observations"),

    ("Complete age-sex combinations",
     len(missing_pairs) == 0
     and len(unexpected_pairs) == 0,
     f"{len(missing_pairs)} missing; "
     f"{len(unexpected_pairs)} unexpected"),

    ("No duplicated age-sex combinations",
     duplicate_count == 0,
     f"{duplicate_count} duplicates"),

    ("No missing values",
     missing_values.sum() == 0,
     f"{missing_values.sum()} missing values"),

    ("Valid mortality probabilities",
     valid_qx,
     "0 <= qx <= 1"),

    ("Nonnegative life-table values",
     valid_life_values,
     "lx >= 0 and ex >= 0"),

    ("Initial survivor counts",
     valid_initial_survivors,
     "100,000 for each schedule"),

    ("Nonincreasing survivor counts",
     monotone_survivors,
     "lx does not increase with age")
]

quality_df = pd.DataFrame(
    checks,
    columns=["Validation check", "Passed", "Details"]
)

quality_df["Status"] = quality_df["Passed"].map(
    {True: "PASS", False: "FAIL"}
)

display(
    quality_df[
        ["Validation check", "Status", "Details"]
    ]
)

# Stop the analysis if any validation check fails
if not quality_df["Passed"].all():
    raise ValueError(
        "Mortality data failed one or more "
        "quality checks."
    )

## Save data and update the report tables

In [ ]:
def write_report_table(frame, filename, *, include_index=False, floatfmt=None):
    """Write an output table that the code-free Quarto report can include."""
    kwargs = {"index": include_index}
    if floatfmt is not None:
        kwargs["floatfmt"] = floatfmt
    (report_results_dir / filename).write_text(
        frame.to_markdown(**kwargs) + "\n", encoding="utf-8"
    )

processed_dir = project_root / "data" / "processed"
processed_dir.mkdir(parents=True, exist_ok=True)
mortality.to_csv(processed_dir / "ssa_mortality_clean.csv", index=False)

preview = mortality.head().rename(columns={
    "age":"Age", "sex":"Sex", "qx":"qₓ", "lx":"lₓ", "ex":"eₓ"
})
write_report_table(preview, "01_cleaned_preview.qmd", floatfmt=".6f")
write_report_table(quality_df[["Validation check","Status","Details"]], "02_quality_checks.qmd")
print("Saved cleaned CSV and updated the first two report tables.")
